<a href="https://colab.research.google.com/github/w2259108/arclight/blob/main/Vincent_Casaceli_Module10_18_Homework03_Ch06_01_CNN_RNN_ATIS_intents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook taken from https://github.com/practical-nlp/practical-nlp-code/blob/master/Ch6/01_CNN_RNN_ATIS_intents.ipynb

## Intent Detection Using CNN & RNN


In this notebook we demonstrate various CNN and RNN architectures for the task of intent detection on the ATIS dataset. The ATIS dataset is a standard benchmark dataset for the task of intent detection. ATIS Stands for Airline Travel Information System. The dataset can we found in the `Data2` folder under the `Data` folder.

In [7]:
import os, shutil
print('cwd:', os.getcwd())
os.makedirs('Data/Data2', exist_ok=True)
for fn in ['atis.train.w-intent.iob', 'atis.test.w-intent.iob']:
    if os.path.exists(fn): shutil.move(fn, 'Data/Data2/' + fn)
if os.path.exists('glove.6B.100d.txt'): shutil.move('glove.6B.100d.txt', 'Data/glove.6B.100d.txt')
print('here:', os.listdir('.'))
print('Data:', os.listdir('Data'))
print('Data2:', os.listdir('Data/Data2'))

cwd: /content
here: ['.config', 'Data', 'sample_data']
Data: ['Data2']
Data2: []


## Imports

In [5]:
#general imports
# cell 5
# run
import os
import sys
import random
random.seed(0) #for reproducability of results

#basic imports
import numpy as np
import pandas as pd


#NN imports
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Input, GlobalMaxPooling1D
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.initializers import Constant

#encoder
from sklearn.preprocessing import LabelEncoder

print("Step #01: imports completed!")



Step #01: imports completed!


## Data Loading
We load the data with the help of a few functions from `utils.py` which is included in this repository's Ch6 folder under folder name "Data".
### Training Data


In [2]:
# cell 7
import pandas as pd
import numpy as np

def get_data(filename):
    df = pd.read_csv(filename,sep=r'\s+',names=['word','label'])  # FIX delim_whitespace was removed in pandas 3.x; sep=r'\s+' is the equivalent
    beg_indices = list(df[df['word'] == 'BOS'].index)+[df.shape[0]]
    sents,labels,intents = [],[],[]
    for i in range(len(beg_indices[:-1])):
        sents.append(df[beg_indices[i]+1:beg_indices[i+1]-1]['word'].values)
        labels.append(df[beg_indices[i]+1:beg_indices[i+1]-1]['label'].values)
        intents.append(df.loc[beg_indices[i+1]-1]['label'])
    return np.array(sents, dtype=object), np.array(labels, dtype=object), np.array(intents, dtype=object)

def get_data2(filename):
    with open(filename) as f:
        contents = f.read()
    sents,labels,intents = [],[],[]
    for line in contents.strip().split('\n'):
        words,labs = [i.split(' ') for i in line.split('\t')]
        sents.append(words[1:-1])
        labels.append(labs[1:-1])
        intents.append(labs[-1])
    return np.array(sents, dtype=object), np.array(labels, dtype=object), np.array(intents, dtype=object)

read_method = {'Data/Data2/atis-2.dev.w-intent.iob':get_data,
               'Data/Data2/atis.train.w-intent.iob':get_data2,
               'Data/Data2/atis.test.w-intent.iob':get_data,
              'Data/Data2/atis-2.train.w-intent.iob':get_data2}

def fetch_data(fname):
    func = read_method[fname]
    return func(fname)

print("Step #02: functions defined!")

Step #02: functions defined!


In [8]:
# cell 8
sents,labels,intents = fetch_data('Data/Data2/atis.train.w-intent.iob')

train_sentences = [" ".join(i) for i in sents]

train_texts = train_sentences
train_labels= intents.tolist()

vals = []

for i in range(len(train_labels)):
    if "#" in train_labels[i]:
        vals.append(i)

for i in vals[::-1]:
    train_labels.pop(i)
    train_texts.pop(i)

print ("Number of training sentences :",len(train_texts))
print ("Number of unique intents :",len(set(train_labels)))

for i in zip(train_texts[:5], train_labels[:5]):
    print(i)

print("Step #03: training data loaded!")

Number of training sentences : 4952
Number of unique intents : 17
('i want to fly from boston at 838 am and arrive in denver at 1110 in the morning', 'atis_flight')
('what flights are available from pittsburgh to baltimore on thursday morning', 'atis_flight')
('what is the arrival time in san francisco for the 755 am flight leaving washington', 'atis_flight_time')
('cheapest airfare from tacoma to orlando', 'atis_airfare')
('round trip fares from pittsburgh to philadelphia under 1000 dollars', 'atis_airfare')
Step #03: training data loaded!


### Testing Data

In [9]:
# cell 10
sents,labels,intents = fetch_data('Data/Data2/atis.test.w-intent.iob')

test_sentences = [" ".join(i) for i in sents]

test_texts = test_sentences
test_labels = intents.tolist()

new_labels = set(test_labels) - set(train_labels)

vals = []

for i in range(len(test_labels)):
    if "#" in test_labels[i]:
        vals.append(i)
    elif test_labels[i] in new_labels:
        print(test_labels[i])
        vals.append(i)

for i in vals[::-1]:
    test_labels.pop(i)
    test_texts.pop(i)

print ("Number of testing sentences :",len(test_texts))
print ("Number of unique intents :",len(set(test_labels)))

for i in zip(test_texts[:5], test_labels[:5]):
    print(i)

print("Step #04: testing data loaded!")

atis_day_name
atis_day_name
Number of testing sentences : 876
Number of unique intents : 15
('i would like to find a flight from charlotte to las vegas that makes a stop in st. louis', 'atis_flight')
('on april first i need a ticket from tacoma to san jose departing before 7 am', 'atis_airfare')
('on april first i need a flight going from phoenix to san diego', 'atis_flight')
('i would like a flight traveling one way from phoenix to san diego on april first', 'atis_flight')
('i would like a flight from orlando to salt lake city for april first on delta airlines', 'atis_flight')
Step #04: testing data loaded!


## Data Preprocessing

In [10]:
# cell 12
MAX_SEQUENCE_LENGTH = 300
MAX_NUM_WORDS = 20000
EMBEDDING_DIM = 100
VALIDATION_SPLIT = 0.3

tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(train_texts)
train_sequences = tokenizer.texts_to_sequences(train_texts) #Converting text to a vector of word indexes
test_sequences = tokenizer.texts_to_sequences(test_texts)
word_index = tokenizer.word_index
print('Step #05: Found %s unique tokens.' % len(word_index))

Step #05: Found 897 unique tokens.


In [11]:
# cell 13
le = LabelEncoder()
le.fit(train_labels)
train_labels = le.transform(train_labels)
test_labels = le.transform(test_labels)

#Converting this to sequences to be fed into neural network. Max seq. len is 1000 as set earlier
 #initial padding of 0s, until vector is of size MAX_SEQUENCE_LENGTH
trainvalid_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)
trainvalid_labels = to_categorical(train_labels)

test_labels = to_categorical(np.asarray(test_labels), num_classes= trainvalid_labels.shape[1])

# split the training data into a training set and a validation set
indices = np.arange(trainvalid_data.shape[0])
np.random.shuffle(indices)
trainvalid_data = trainvalid_data[indices]
trainvalid_labels = trainvalid_labels[indices]
num_validation_samples = int(VALIDATION_SPLIT * trainvalid_data.shape[0])
x_train = trainvalid_data[:-num_validation_samples]
y_train = trainvalid_labels[:-num_validation_samples]
x_val = trainvalid_data[-num_validation_samples:]
y_val = trainvalid_labels[-num_validation_samples:]
#This is the data we will use for CNN and RNN training
print('Step #06: Splitting the train data into train and valid is done')

Step #06: Splitting the train data into train and valid is done


## Modeling
### Embedding Matrix
We need to prepare our embedding.

In [12]:
# cell 15
print('Preparing embedding matrix.')

def process_glove_line(line, dim):
    word = None
    embedding = None

    try:
        splitLine = line.split()
        word = " ".join(splitLine[:len(splitLine)-dim])
        embedding = np.array([float(val) for val in splitLine[-dim:]])
    except:
        print(line)

    return word, embedding

def load_glove_model(glove_filepath, dim):
    with open(glove_filepath, encoding="utf8" ) as f:
        content = f.readlines()
        model = {}
        for line in content:
            word, embedding = process_glove_line(line, dim)
            if embedding is not None:
                model[word] = embedding
        return model

# first, build index mapping words in the embeddings set
# to their embedding vector

# Download GloVe 6B from here: https://nlp.stanford.edu/projects/glove/
BASE_DIR = 'Data'
#GLOVE_DIR = os.path.join(BASE_DIR, 'glove.6B')
GLOVE_DIR = BASE_DIR


#embeddings_index = {}
#with open(os.path.join(GLOVE_DIR, 'glove.6B.100d.txt'), encoding="utf-8") as f:
#    for line in f:
#        values = line.split()
#        word = values[0]
#        coefs = np.asarray(values[1:], dtype='float32')
#        embeddings_index[word] = coefs

embeddings_index = load_glove_model('Data/glove.6B.100d.txt', 100)

print('Found %s word vectors in Glove embeddings.' % len(embeddings_index))
#print(embeddings_index["google"])

# prepare embedding matrix - rows are the words from word_index, columns are the embeddings of that word from glove.
num_words = min(MAX_NUM_WORDS, len(word_index)) + 1
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
for word, i in word_index.items():
    if i > MAX_NUM_WORDS:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

# load these pre-trained word embeddings into an Embedding layer
# note that we set trainable = False so as to keep the embeddings fixed
embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer=Constant(embedding_matrix),
                            # FIX input_length removed -- Keras 3 rejects this argument; sequence length is inferred from the input
                            trainable=False)
print("Step #07: Preparing of embedding matrix is done")

Preparing embedding matrix.
touchy -0.78137 0.25798 0.72795 0.27161 -0.68758 -0.050562 -0.60473 -0.85951 0.8265 -0.0070202 -0.27578 0.19527 0.10534 -0.42253 0.12044 0.11334 0.19528 -0.41158 0.67818 -0.73836 0.2172 0.20338 0.34902 0.0049379 0.58495 -0.23907 -0.1631 0.4193 -0.55376 0.49873 -0.93183 -0.23327 1.0704 0.29867 -0.32497 0.75211 -0.49888 0.3341 -0.087921 0.04276 0.31395 -0.2038 0.44
Found 30554 word vectors in Glove embeddings.
Step #07: Preparing of embedding matrix is done


### CNN with Pre-Trained Embeddings

In [13]:
# cell 17
print('Define a 1D CNN model.')

cnnmodel = Sequential()
cnnmodel.add(embedding_layer)
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(GlobalMaxPooling1D())
cnnmodel.add(Dense(128, activation='relu'))
cnnmodel.add(Dense(len(trainvalid_labels[0]), activation='softmax'))

cnnmodel.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['acc'])

cnnmodel.summary()

#Train the model. Tune to validation set.
cnnmodel.fit(x_train, y_train,
          batch_size=128,
          epochs=1, validation_data=(x_val, y_val))
#Evaluate on test set:
score, acc = cnnmodel.evaluate(test_data, test_labels)
print('Step #08: Test accuracy with CNN:', acc)

Define a 1D CNN model.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 287ms/step - acc: 0.7095 - loss: 1.1634 - val_acc: 0.7475 - val_loss: 1.0154
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - acc: 0.7215 - loss: 1.1558
Step #08: Test accuracy with CNN: 0.7214611768722534


### CNN-Embedding Layer
Here, we train a CNN model with an embedding layer which is being trained on the fly instead of using the pre-trained embeddings.

In [14]:
# cell 19
print("Defining and training a CNN model, training embedding layer on the fly instead of using pre-trained embeddings")
cnnmodel = Sequential()
cnnmodel.add(Embedding(MAX_NUM_WORDS, 128))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(MaxPooling1D(5))
cnnmodel.add(Conv1D(128, 5, activation='relu'))
cnnmodel.add(GlobalMaxPooling1D())
cnnmodel.add(Dense(128, activation='relu'))
cnnmodel.add(Dense(len(trainvalid_labels[0]), activation='softmax'))

cnnmodel.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['acc'])

cnnmodel.summary()

#Train the model. Tune to validation set.
cnnmodel.fit(x_train, y_train,
          batch_size=128,
          epochs=1, validation_data=(x_val, y_val))
#Evaluate on test set:
score, acc = cnnmodel.evaluate(test_data, test_labels)
print('Step #09: Test accuracy with CNN:', acc)


Defining and training a CNN model, training embedding layer on the fly instead of using pre-trained embeddings


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

28/28 ━━━━━━━━━━━━━━━━━━━━ 20s 469ms/step - acc: 0.7133 - loss: 1.4820 - val_acc: 0.7475 - val_loss: 1.3660
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - acc: 0.7215 - loss: 1.5039
Step #09: Test accuracy with CNN: 0.7214611768722534


### RNN-Embedding Layer
Here, we train a RNN model with an embedding layer which is being trained on the fly instead of using the pre-trained embeddings.

In [17]:
# cell 21
print("Defining and training an LSTM model, training embedding layer on the fly")

#modified from:

rnnmodel = Sequential()
rnnmodel.add(Embedding(MAX_NUM_WORDS, 128))
rnnmodel.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
rnnmodel.add(Dense(len(trainvalid_labels[0]), activation='sigmoid'))
rnnmodel.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

rnnmodel.summary()

print('Training the RNN')
rnnmodel.fit(x_train, y_train,
          batch_size=32,
          epochs=10,
          validation_data=(x_val, y_val))
score, acc = rnnmodel.evaluate(test_data, test_labels,
                            batch_size=32)
print('Step #10: Test accuracy with RNN:', acc)


Defining and training an LSTM model, training embedding layer on the fly


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Training the RNN
Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.7294 - loss: 0.1543 - val_accuracy: 0.7475 - val_loss: 0.0984
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.7372 - loss: 0.1004 - val_accuracy: 0.7475 - val_loss: 0.0942
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.7459 - loss: 0.0881 - val_accuracy: 0.7886 - val_loss: 0.0723
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - accuracy: 0.8070 - loss: 0.0678 - val_accuracy: 0.8539 - val_loss: 0.0567
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8566 - loss: 0.0525 - val_accuracy: 0.8774 - val_loss: 0.0466
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8820 - loss: 0.0423 - val_accuracy: 0.8929 - val_loss: 0.0402
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.8892 - loss: 0.0369 - val_accuracy: 0.9037 - val_loss: 0.0370
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9068 - loss: 

### LSTM with Pre-Trained Embeddings

In [18]:
# cell 23
print("Defining and training an LSTM model, using pre-trained embedding layer")

#modified from:

rnnmodel2 = Sequential()
rnnmodel2.add(embedding_layer)
rnnmodel2.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
rnnmodel2.add(Dense(len(trainvalid_labels[0]), activation='sigmoid'))
rnnmodel2.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

rnnmodel2.summary()

print('Training the RNN')
rnnmodel2.fit(x_train, y_train,
          batch_size=32,
          epochs=1,
          validation_data=(x_val, y_val))
score, acc = rnnmodel2.evaluate(test_data, test_labels,
                            batch_size=32)
print('Step #11: Test accuracy with RNN:', acc)


Defining and training an LSTM model, using pre-trained embedding layer


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 100)       │        89,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,800 (350.78 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 89,800 (350.78 KB)

Training the RNN
109/109 ━━━━━━━━━━━━━━━━━━━━ 105s 936ms/step - accuracy: 0.7286 - loss: 0.1430 - val_accuracy: 0.7475 - val_loss: 0.0925
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 192ms/step - accuracy: 0.7215 - loss: 0.1056
Step #11: Test accuracy with RNN: 0.7214611768722534
